In [ ]:
"""Training entrypoint
"""

import os

import torch

from torch import optim
from torch.utils.data import DataLoader
from torch import nn


"""VGG11 encoder
"""
import os
import random
import xml.etree.ElementTree as ET

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

from typing import Dict, Tuple, Union

import torch.nn.functional as F

In [ ]:




class OxfordIIITPetRawDataset(Dataset):
    """Loads channel-first images, class-index labels, masks, and bboxes."""

    NUM_CLASSES = 37

    def __init__(self, root_dir: str, mode: str = "train", image_size=(224, 224)):
        self.root_dir = root_dir
        self.mode = mode
        self.image_size = image_size

        self.images_dir = os.path.join(root_dir, "images", "images")
        self.annotations_dir = os.path.join(root_dir, "annotations", "annotations")
        self.masks_dir = os.path.join(self.annotations_dir, "trimaps")
        self.xmls_dir = os.path.join(self.annotations_dir, "xmls")

        self.samples = self._load_split_samples()

    def _mask_path(self, image_id: str) -> str:
        candidates = [
            os.path.join(self.masks_dir, f"{image_id}.png"),
            os.path.join(self.masks_dir, f"._{image_id}.png"),
        ]

        for path in candidates:
            if os.path.exists(path):
                return path

        raise FileNotFoundError(f"No trimap found for {image_id}")

    def _split_file_path(self) -> str:
        split_files = {
            "train": "trainval.txt",
            "val": "trainval.txt",
            "test": "test.txt",
        }
        if self.mode not in split_files:
            raise ValueError("mode must be 'train', 'val' or 'test'")
        return os.path.join(self.annotations_dir, split_files[self.mode])

    def _partition_trainval_samples(self, samples, val_ratio=0.2, random_state=42):
        indices = list(range(len(samples)))
        rng = random.Random(random_state)
        rng.shuffle(indices)

        val_size = int(len(indices) * val_ratio)
        val_indices = set(indices[:val_size])

        train_samples = [samples[i] for i in range(len(samples)) if i not in val_indices]
        val_samples = [samples[i] for i in range(len(samples)) if i in val_indices]
        return train_samples, val_samples

    def _load_split_samples(self):
        split_file = self._split_file_path()
        samples = []

        with open(split_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue

                image_id, class_id, species, breed_id = line.split()
                xml_path = os.path.join(self.xmls_dir, f"{image_id}.xml")
                if self.mode in {"train", "val"} and not os.path.exists(xml_path):
                    continue

                samples.append(
                    {
                        "image_id": image_id,
                        "image_path": os.path.join(self.images_dir, f"{image_id}.jpg"),
                        "mask_path": self._mask_path(image_id),
                        "xml_path": xml_path,
                        "class_id": int(class_id) - 1,
                        "species": int(species),
                        "breed_id": int(breed_id),
                    }
                )

        if self.mode in {"train", "val"}:
            train_samples, val_samples = self._partition_trainval_samples(samples)
            return train_samples if self.mode == "train" else val_samples

        return samples

    def _load_bbox(self, xml_path: str, image_shape) -> tuple[np.ndarray, bool]:
        height, width = image_shape[:2]
        if not os.path.exists(xml_path):
            return np.array([0.0, 0.0, float(width), float(height)], dtype=np.float32), False

        root = ET.parse(xml_path).getroot()
        bndbox = root.find(".//bndbox")
        if bndbox is None:
            return np.array([0.0, 0.0, float(width), float(height)], dtype=np.float32), False

        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)
        return np.array([xmin, ymin, xmax, ymax], dtype=np.float32), True

    def _resize_sample(self, image, mask, bbox):
        original_height, original_width = image.shape[:2]
        target_width, target_height = self.image_size

        image = Image.fromarray(image).resize((target_width, target_height), Image.BILINEAR)
        mask = Image.fromarray(mask).resize((target_width, target_height), Image.NEAREST)

        bbox = bbox.astype(np.float32).copy()
        bbox[[0, 2]] *= target_width / original_width
        bbox[[1, 3]] *= target_height / original_height

        return np.array(image), np.array(mask), bbox

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        image = np.array(Image.open(sample["image_path"]).convert("RGB"))
        mask = np.array(Image.open(sample["mask_path"]))
        bbox, has_bbox = self._load_bbox(sample["xml_path"], image.shape)
        image, mask, bbox = self._resize_sample(image, mask, bbox)
        image = torch.from_numpy(image.astype(np.float32) / 255.0).permute(2, 0, 1)
        label = torch.tensor(sample["class_id"], dtype=torch.long)

        return {
            "image": image,
            "mask": mask,
            "label": label,
            "bbox": bbox
            # "split": self.mode,
            # "class_id": sample["class_id"],
            # "image_id": sample["image_id"],
            # "species": sample["species"],
            # "breed_id": sample["breed_id"],
            # "has_bbox": has_bbox,
        }


In [ ]:


class CustomDropout(nn.Module):
    """Custom Dropout layer.
    """

    def __init__(self, p: float = 0.5):
        """
        Initialize the CustomDropout layer.

        Args:
            p: Dropout probability.
        """
        super().__init__()
        self.requires_grad_ = False
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the CustomDropout layer.

        Args:
            x: Input tensor for shape [B, C, H, W].

        Returns:
            Output tensor.
        """
        # We are using pytorch's built in training attribute assigned to the Module class
        if not self.training:
            return x
        self.mask = torch.rand_like(x) > self.p
        return x * self.mask / (1 - self.p)         # using inverted dropout 

In [ ]:


class VGG11Encoder(nn.Module):
    """VGG11-style encoder with optional intermediate feature returns.
    """

    def __init__(self, in_channels: int = 3):
        """Initialize the VGG11Encoder model."""
        super().__init__()
        self.dropout = CustomDropout(p=0.5)
        self.in_channels = in_channels
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(512*7*7, 4096) # Final feature map size is 512x7x7 after 5 maxpools
        self.fc2 = nn.Linear(4096, 4096)
        self.fc3 = nn.Linear(4096, 1000)

    def forward(
        self, x: torch.Tensor, return_features: bool = False
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        """Forward pass.

        Args:
            x: input image tensor [B, 3, H, W].
            return_features: if True, also return skip maps for U-Net decoder.

        Returns:
            - if return_features=False: bottleneck feature tensor.
            - if return_features=True: (bottleneck, feature_dict).
        """
        features = {}
        x = self.conv1(x)
        features['conv1'] = x
    
        x = self.maxpool(x)
        x = self.conv2(x)
        features['conv2'] = x
        x = self.maxpool(x)
        activations_3 = x
        x = self.conv3(x)
        features['conv3'] = x
        x = self.maxpool(x)
        x = self.conv4(x)
        features['conv4'] = x
        x = self.maxpool(x)
        x = self.conv5(x)
        features['conv5'] = x
        x = self.maxpool(x)
        x = torch.flatten(x, 1) 
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        bottleneck = F.relu(self.fc3(x))
        if return_features:
            return bottleneck, features
        else:
            return bottleneck , activations_3

In [ ]:



class VGG11Classifier(nn.Module):
    """Full classifier = VGG11Encoder + ClassificationHead."""

    def __init__(self, num_classes: int = 37, in_channels: int = 3, dropout_p: float = 0.5):
        """
        Initialize the VGG11Classifier model.
        Args:
            num_classes: Number of output classes.
            in_channels: Number of input channels.
            dropout_p: Dropout probability for the classifier head.
        """
        super().__init__()
        self.VGG11enc = VGG11Encoder(in_channels) 
        self.classification_head = nn.Linear(1000, num_classes) 
        


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass for classification model.
        Args:
            x: Input tensor of shape [B, in_channels, H, W].
        Returns:
            Classification logits [B, num_classes].
        """
        x, activation = self.VGG11enc(x)
        x = self.classification_head(x)
        return x, activation


In [ ]:


def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].float()
            labels = batch["label"].long()

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            total_correct += (predictions == labels).sum().item()
            total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy




def train_classifier(num_epochs,continue_train=False):
    print("Loading datasets...")
    train_dataset = OxfordIIITPetRawDataset(root_dir="../dataset/", mode="train")
    val_dataset = OxfordIIITPetRawDataset(root_dir="../dataset/", mode="val")
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4, persistent_workers=True)
    print("Datasets loaded.")
    model = VGG11Classifier(num_classes=37)
    best_model_path = "best_model_classifier.pt"
    if continue_train:
        model.load_state_dict(torch.load(best_model_path))
        print(f"Loaded weights from {best_model_path} for continued training.")
    

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0.0
    train_loss = []
    val_loss = []
    activations_3 = []  # List to store activations from conv3 for each epoch
    
    for epoch in range(num_epochs):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_samples = 0

        for batch in train_loader:
            images = batch["image"].float()
            labels = batch["label"].long()

            optimizer.zero_grad()
            outputs, activation = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * images.size(0)
            activations_3.append(activation)
            predictions = outputs.argmax(dim=1)
            train_correct += (predictions == labels).sum().item()
            train_samples += images.size(0)


        train_loss = train_loss_sum / train_samples
        train_acc = train_correct / train_samples
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        train_loss.append(train_loss)
        val_loss.append(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)


        print(
            f"Epoch {epoch + 1}: "
            f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f}, "
            f"val_acc={val_acc:.4f}, val_loss={val_loss:.4f}, "
            f"best_val_acc={best_val_acc:.4f}"
        )
    np.save("train_loss.npy", np.array(train_loss))
    np.save("val_loss.npy", np.array(val_loss))
    np.save("activations_3.npy", np.array(activations_3))



torch.set_num_threads(os.cpu_count())        
torch.set_num_interop_threads(os.cpu_count())
train_classifier(50,continue_train=True)
